In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec
from matplotlib import rc

In [ ]:
findings = pd.read_csv("validation_proof_of_genuity.csv", index_col=0)
findings

In [ ]:
mask = findings["functional_tp"] == True
tp_findings = findings[mask]
fp_findings = findings[~mask]

print(findings[findings["juiciness_bucket"] == 11])

# there are only a hand full of findings in bucket 11, move them to 10
tp_findings.loc[tp_findings["juiciness_bucket"] == 11, "juiciness_bucket"] = 10

In [ ]:
def get_ratios(dedup=False):
    # proof of intended behavior is present
    def get_ratio(mask, dedup=False):
        yes = tp_findings[mask]
        no = tp_findings[~mask]
        if dedup:
            yes = yes.drop_duplicates(["__bug_id", "source_fs"])
            no = no.drop_duplicates(["__bug_id", "source_fs"])
        yes = yes.groupby(["juiciness_bucket","source_fs"])["manufacturer"].count().groupby("juiciness_bucket").sum().to_frame().rename(columns={"manufacturer": "count"})
        no = no.groupby(["juiciness_bucket","source_fs"])["manufacturer"].count().groupby("juiciness_bucket").sum().to_frame().rename(columns={"manufacturer": "count"})
        
        ratio = yes["count"].div(yes["count"].add(no["count"], fill_value=0), fill_value=0)
        return ratio
    
    
    labels = [
        "\\textbf{E}vidence of Intended Behavior Avail.",
        "\\textbf{C}onvenience Wrapper Usage",
        "\\textbf{K}nown Software Packages",
        "\\textbf{P}rocess Control (CWE-114)",
        "\\textbf{A}rgument Injection (CWE-88)",
        "\\textbf{O}S Command Injection (CWE-78)"
    ]
    
    masks = [
        tp_findings["intended_behavior_documented"] == True,
        ~tp_findings["sink"].str.contains("exec"),
        tp_findings["verified_oss_package"] != "-",
        tp_findings["injection_class"] == "PD",
        tp_findings["injection_class"] == "AI",
        tp_findings["injection_class"] == "SI",
    ]

    ratios = pd.DataFrame({}, index=list(range(0,11)))

    ratios.index.name = "juiciness_bucket"

    for label, mask in zip(labels, masks):
        ratios[label] = get_ratio(mask, dedup).fillna(0.0)

    return ratios.dropna()

ratios = get_ratios(dedup=True)
ratios

In [ ]:
fig = plt.figure(
    figsize=(8, 6),
)

gs = GridSpec(2, 1, height_ratios=[1, 3])

ax_total = fig.add_subplot(gs[0])
ax_line = fig.add_subplot(gs[1], sharex=ax_total)

x = np.arange(1, 11)

cmap = sns.color_palette("colorblind")

# dist plot total / dedup

fp_count = fp_findings["juiciness_bucket"].value_counts().fillna(0)
tp_count = tp_findings["juiciness_bucket"].value_counts().fillna(0)
counts_total = fp_count.add(tp_count, fill_value=0).to_frame()


ax_line.axvline(6, linestyle="-", color="black", linewidth=3)
ax_line.axvline(7, linestyle="-", color="black", linewidth=3)
ax_line.set_axisbelow(True)
ax_line.grid(True, zorder=1)


ax_total.set_axisbelow(True)
ax_total.axvline(6, linestyle="-", color="black", linewidth=3, zorder=10)
ax_total.axvline(7, linestyle="-", color="black", linewidth=3, zorder=10)
ax_total.grid(True)

ax_total.bar(x, counts_total["count"], color=cmap[1], edgecolor="black", linewidth=1, label="All Findings (TP+FP; $\\Sigma = 8,702)$", zorder=11)

fp_count = fp_findings.drop_duplicates(["__bug_id", "source_fs"])["juiciness_bucket"].value_counts().fillna(0)
tp_count = tp_findings.drop_duplicates(["__bug_id", "source_fs"])["juiciness_bucket"].value_counts().fillna(0)
counts_dedup = fp_count.add(tp_count, fill_value=0).to_frame()
ax_total.bar(x, counts_dedup["count"], color=cmap[-1], edgecolor="black", linewidth=1, label="Deduplicated (TP; $\\Sigma = 4,357$)", zorder=12)

handles, labels = ax_total.get_legend_handles_labels()


ax_total.legend(
    handles,
    labels,
    loc="upper center",
    fontsize=14,
    ncol=2,
    bbox_to_anchor=(0.45, 1.45),
)

line_palette = [cmap[0], "black", cmap[2], "gray", cmap[5], cmap[3],  "black",  cmap[1], cmap[3]]

ratios.plot(
    kind="line",
    markeredgecolor="black",
    markersize=8,
    markeredgewidth=1,
    style=["-o", "-o", "-o", "-s", "-s", "-s"],
    linewidth=2,
    color=line_palette,
    ax=ax_line, grid=True, legend=False)

handles, labels = ax_line.get_legend_handles_labels()

group_enrichment = [0, 2, 1]
group_bugs = [3, 4, 5]

ax_total.text(6.5, 2100, "Overlap", color="black", fontweight="heavy", fontsize=15, ha="center", va="top", rotation=45)



ax_line.add_artist(ax_line.legend(
    [handles[i] for i in group_enrichment],
    [labels[i] for i in group_enrichment],
    title="Findings: Properties",
    loc="lower left",
    fontsize=14,
    #ncol=2,
    bbox_to_anchor=(-0.145, -0.58),
))

ax_line.legend(
    [handles[i] for i in group_bugs],
    [labels[i] for i in group_bugs],
    title="Findings: Classification",
    fontsize=14,
    loc="lower left",
    ncol=1,
    bbox_to_anchor=(0.45, -0.58),
)


ax_total.set_yticks([0, 1000, 2000])

ax_total.set_ylabel("Findings [\\#]")
ax_total.set_xlim(0.6, 10.5)
ax_line.set_yticks([0.1 * x for x in range(0, 11)])
ax_line.set_xticks(range(1, 11))
ax_line.set_xlabel("Buckets [Juiciness]")
ax_line.set_ylabel("Property Share [Rel.]")

plt.savefig("juicepress_main_results_MOCK.pdf", bbox_inches="tight")

plt.show()
# line plot

In [ ]:
share_df = tp_findings.groupby(['juiciness_bucket', '__urgency_class']).size().unstack(fill_value=0)
share_df = tp_findings.drop_duplicates(subset=["__bug_id", "source_fs"]).groupby(['juiciness_bucket', '__urgency_class']).size().unstack(fill_value=0)
share_df = share_df.div(share_df.sum(axis=1), axis=0)  # convert counts to percentages
share_df = share_df[["NONE"]].rename(columns={"NONE": "Informative"})
share_df.index = share_df.index.astype(int)
print(share_df)

In [ ]:
ax = share_df.plot(kind='bar', width=0.6, stacked=True, figsize=(10,3), edgecolor="black", linewidth=1, color=["#b9b9b9", cmap[2], cmap[1], cmap[3], cmap[4]], legend=False)

ax.legend(
    fontsize=14,
    loc="upper center",
    ncol=5,
    bbox_to_anchor=(0.5, 1.2),  
)

ax.set_axisbelow(True)
ax.grid(True, zorder=-4)
ax.tick_params(axis="x", labelrotation=0)
ax.set_yticks([0.1 * x for x in range(0, 11)])
ax.set_ylabel("Share [Rel.]")
ax.set_xlabel("Bucket [Juiciness]")
ax.axvline(4.5, linestyle="-", color="black", linewidth=3)
ax.axvline(6.5, linestyle="-", color="black", linewidth=3)


plt.savefig("juicepress_criticality_MOCK.pdf", bbox_inches="tight")
plt.show()